 # LDA Gensim

In [1]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity  
import numpy as np


In [3]:
# Download NLTK resources if not already present
nltk.download('punkt')
nltk.download('stopwords')

# Load dataset
df = pd.read_csv("kompas_politik_bola.csv")

# Hapus baris kosong awal
df = df.dropna(subset=['ringkasan'])
df = df[df['ringkasan'].str.strip() != ""]
print(f"📦 Dataset setelah menghapus baris kosong awal: {len(df)} baris")

# Gunakan Sastrawi untuk stemming
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Stopwords Bahasa Indonesia
stop_words = set(stopwords.words('indonesian'))
exception_words = set()

# --- Fungsi untuk membersihkan watermark kompas.com dan kota ---
def clean_kompas_pattern(text):
    if not isinstance(text, str):
        return ""
    # Pola baru: menangani baik ada kota maupun tidak
    # Contoh yang dihapus:
    # "Jakarta, Kompas.com -", "Kompas.com—", "Kompas.com:", "Surabaya kompas.com–"
    pattern = r'^(?:[A-Z][a-zA-Z\s]*,?\s*)?kompas\.com[—\-–:]*\s*'
    text = re.sub(pattern, '', text, flags=re.IGNORECASE).strip()
    return text

# --- Fungsi preprocessing utama ---
def preprocess_text(text):
    text = clean_kompas_pattern(text)  # bersihkan watermark dan kota
    
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    stemmed_tokens = []
    for w in tokens:
        if w in exception_words:
            stemmed_tokens.append(w)
        else:
            stem = stemmer.stem(w)
            if len(stem) < 3:
                stemmed_tokens.append(w)
            else:
                stemmed_tokens.append(stem)
    return " ".join(stemmed_tokens).strip()

# --- Jalankan preprocessing ---
text_col = "ringkasan"
df['preprocessed'] = df[text_col].astype(str).apply(preprocess_text)

# --- Hapus baris kosong setelah preprocessing ---
before = len(df)
df = df[df['preprocessed'].str.strip() != ""]
after = len(df)

print(f"📊 Data sebelum pembersihan: {before} baris")
print(f"📉 Data sesudah pembersihan: {after} baris")
print(f"🗑️ Baris kosong yang dihapus: {before - after}")

# --- Simpan hasil ---
df.to_csv("prepeocessed.csv", index=False)
print("✅ File prepeocessed.csv berhasil disimpan.")

# --- Tampilkan contoh hasil ---
print("\n🧾 Contoh hasil pembersihan:")
print(df[['ringkasan', 'preprocessed']].head(5))


[nltk_data] Downloading package punkt to C:\Users\Mukti
[nltk_data]     Hendra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Mukti
[nltk_data]     Hendra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


📦 Dataset setelah menghapus baris kosong awal: 624 baris
📊 Data sebelum pembersihan: 624 baris
📉 Data sesudah pembersihan: 624 baris
🗑️ Baris kosong yang dihapus: 0
✅ File prepeocessed.csv berhasil disimpan.

🧾 Contoh hasil pembersihan:
                                           ringkasan  \
0  KOMPAS.com- Ketua Komisi XI DPR RI, MukhamadMi...   
1  KOMPAS.com- Wakil Ketua (Waka) Badan Legislasi...   
2  KOMPAS.com- Ketua Dewan Perwakilan Rakyat (DPR...   
3  KOMPAS.com– Dewan Perwakilan Rakyat (DPR) Repu...   
4  KOMPAS.com– Dalam rangka memperingati Maulid N...   

                                        preprocessed  
0  ketua komisi dpr mukhamadmisbakhunmenegaskan m...  
1  wakil ketua waka badan legislasi baleg dewan w...  
2  ketua dewan wakil rakyat dpr ripuan maharanime...  
3  dewan wakil rakyat dpr republik indonesia paka...  
4  rangka ingat maulid nabi muhammad saw forum ko...  


In [7]:
import pandas as pd
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

# Baca data yang sudah diproses
df = pd.read_csv("prepeocessed.csv")

# Pastikan kolom 'preprocessed' tidak kosong
df = df[df['preprocessed'].notnull() & (df['preprocessed'].str.strip() != "")]
texts = df['preprocessed'].astype(str).tolist()

# Tokenisasi (karena kolom sudah bersih)
tokenized_texts = [t.split() for t in texts]

# Buat dictionary dan corpus untuk Gensim
id2word = corpora.Dictionary(tokenized_texts)
corpus = [id2word.doc2bow(text) for text in tokenized_texts]

# Tentukan jumlah topik
num_topics = 22

# Latih model LDA
lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=num_topics,
    random_state=42,
    passes=20,
    alpha='auto',
    per_word_topics=True
)

# Proporsi Kata dalam Topik
topics = lda_model.show_topics(num_topics=num_topics, num_words=10, formatted=False)

topic_word_data = []
for topic_no, words in topics:
    for word, prob in words:
        topic_word_data.append([topic_no + 1, word, prob])

df_topic_words = pd.DataFrame(topic_word_data, columns=["Topik", "Kata", "Proporsi"])
print("\n🧾 Proporsi Kata dalam Topik:")
print(df_topic_words.head(20))

# Proporsi Topik dalam Dokumen
doc_topic_data = []
for i, row in enumerate(lda_model[corpus]):
    for topic_no, prob in row[0]:
        doc_topic_data.append([i + 1, topic_no + 1, prob])

df_doc_topics = pd.DataFrame(doc_topic_data, columns=["Dokumen", "Topik", "Proporsi"])
print("\n📊 Proporsi Topik dalam Dokumen:")
print(df_doc_topics.head(20))

# Coherence Score (untuk evaluasi jumlah topik)
coherence_model = CoherenceModel(model=lda_model, texts=tokenized_texts, dictionary=id2word, coherence='c_v')
coherence = coherence_model.get_coherence()
print(f"\n📈 Coherence Score (num_topics={num_topics}): {coherence:.4f}")

# (Opsional) Simpan hasil ke file CSV
df_topic_words.to_csv("proporsi_kata_dalam_topik.csv", index=False)
df_doc_topics.to_csv("proporsi_topik_dalam_dokumen.csv", index=False)
print("\n✅ Hasil disimpan sebagai CSV.")



🧾 Proporsi Kata dalam Topik:
    Topik       Kata  Proporsi
0       1     tambah  0.023640
1       1      paket  0.023639
2       1     bansos  0.023639
3       1      beras  0.023639
4       1        isi  0.023639
5       1  indonesia  0.019724
6       1      ketua  0.018401
7       1       atur  0.013158
8       1      atlet  0.013158
9       1       skor  0.013158
10      2    jakarta  0.053768
11      2     daerah  0.028664
12      2     rakyat  0.021182
13      2      sabtu  0.019253
14      2      wakil  0.017094
15      2     bangun  0.015421
16      2     khusus  0.015130
17      2      ketua  0.013519
18      2      dewan  0.013517
19      2        dpr  0.013512

📊 Proporsi Topik dalam Dokumen:
    Dokumen  Topik  Proporsi
0         1     20  0.990661
1         2     14  0.988771
2         3     16  0.987472
3         4     15  0.982933
4         5      9  0.987832
5         6      7  0.987604
6         7     15  0.989758
7         8     16  0.983651
8         9      1  0.984